# 🚀 TriP: Triple-Prompt Framework in Continual Graph Learning
### *Aligning Pre-training and Class-Incremental Objectives in Continual Graph Learning*
*(Neural Networks Journal, Elsevier)*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abhireddy28/TriP-Continual-Graph-Learning/blob/main/notebooks/TriP_Colab_Runner.ipynb)

---
### **Overview of Key TriP Components:**
1. **Self-Supervised Pre-training:** Trains GNN backbone on Task 1 with random edge dropout & feature masking, then **permanently freezes** it.
2. **Feature Prompt (GPF-plus):** $\hat{\mathbf{x}}_v = \mathbf{x}_v + \mathbf{p}_v$ adapts the graph node feature space.
3. **Class Prompt:** Feature-wise gating $\tilde{\mathbf{h}}_v = \mathbf{h}_v \odot \mathbf{q}_t^{cls}$ enhances category separability.
4. **Unified Prompt Template:** Aligns downstream continual classification with prototype-based contrastive loss (Eq. 9-14).
5. **Zero-forgetting Task Querying:** Queries test graphs against task prototypes $\mathbf{m}_t$ to select prompts without ground-truth task labels.

## 🛠️ Step 1: Environment Setup & GPU Check
Ensure you have a GPU runtime enabled (`Runtime` > `Change runtime type` > `T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 📥 Step 2: Clone Repository & Install Dependencies
This pulls the complete TriP framework and installs PyTorch Geometric & OGB.

In [ ]:
import os
if not os.path.exists("TriP-Continual-Graph-Learning"):
    !git clone https://github.com/Abhireddy28/TriP-Continual-Graph-Learning.git
    %cd TriP-Continual-Graph-Learning
else:
    %cd TriP-Continual-Graph-Learning
    !git pull

# Install dependencies automatically
!pip install -q torch-geometric ogb seaborn pyyaml

## 🧪 Step 3: Run TriP on CoraFull Benchmark (Automated Download & Training)
The dataset downloads automatically from PyTorch Geometric. We test with 5 continual tasks for a fast interactive demo (or set `max_tasks=None` for all 35 tasks).

In [ ]:
from datasets.cglb_dataset import CGLBDatasetManager
from models.trip_model import TriPFramework
from trainers.pretrainer import SelfSupervisedPretrainer
from trainers.continual_trainer import ContinualTrainer
from evaluate.metrics import plot_performance_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Automatically load CoraFull and build continual tasks
manager = CGLBDatasetManager(dataset_name="CoraFull", data_dir="./data", classes_per_task=2)
# Set max_tasks=5 for rapid viva demo (~2 mins), or max_tasks=None for full 35 tasks
data_dict = manager.build_continual_tasks(max_tasks=5)

x = data_dict["x"].to(device)
edge_index = data_dict["edge_index"].to(device)
y = data_dict["y"].to(device)
tasks = data_dict["tasks"]

# 2. Initialize TriP Framework
model = TriPFramework(
    in_features=data_dict["num_features"],
    hidden_dim=256,
    backbone_type="GCN",
    prompt_k=2,
    temperature=0.5,
    alpha=0.4,
    device=device
).to(device)

# 3. Self-Supervised Pre-training on Task 1
pretrainer = SelfSupervisedPretrainer(model, lr=0.001, epochs=50, edge_drop_p=0.2, feat_mask_p=0.2)
pretrainer.train(x, edge_index, tasks[0]["train_nodes"].to(device))

# 4. Class-Incremental Continual Graph Learning across tasks
trainer = ContinualTrainer(model, tasks, lr=0.005, epochs=40)
results = trainer.train_continual(x, edge_index, y)

## 📊 Step 4: Visualize Performance Matrix Heatmap (Figure 3 in Paper)
Shows the accuracy on task $j$ after learning up to task $i$. High values across columns demonstrate negligible forgetting!

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

plot_performance_matrix(results["accuracy_matrix"], dataset_name="CoraFull", save_path="results/corafull_matrix.png")
display(Image("results/corafull_matrix.png"))

## 🏆 Step 5: Research Paper Benchmark Comparison (Table 2 in Paper)
Let's compare TriP's reported performance against standard SOTA baselines:

In [ ]:
import pandas as pd

benchmark_data = {
    "Method": ["Fine-tune", "EWC", "ER-GNN", "SSM", "CaT", "DeLoMe", "TPP", "TAAM", "MIP", "TriP (Paper)", "TriP (Our Run)"],
    "Category": ["Naive", "Regularization", "Replay", "Replay", "Condensation", "Replay", "Prompt", "Adapter", "Prompt", "Triple-Prompt", "Triple-Prompt"],
    "CoraFull AA (%)": [2.82, 54.21, 34.22, 78.02, 79.07, 82.05, 94.39, 92.50, 94.50, 95.13, f"{results['average_accuracy']:.2f}"],
    "CoraFull AF (%)": [-13.07, -26.54, -29.69, -6.31, -10.80, -2.44, 0.00, 0.00, 0.00, -0.02, f"{results['average_forgetting']:.2f}"]
}

df = pd.DataFrame(benchmark_data)
display(df)

## 🎓 Viva & Faculty Explanation Quick-Reference

- **What is Catastrophic Forgetting?** When learning new categories (e.g. Mechanical/Civil), conventional neural networks overwrite the parameter weights learned for older categories (e.g. CSE/ECE).
- **Why Freeze the GNN Backbone?** Freezing eliminates catastrophic forgetting entirely in the backbone feature extractor. Only lightweight prompts adapt to new tasks.
- **Why Three Prompts (TriP)?**
  1. *Feature Prompt:* Adjusts input node space without modifying graph topology.
  2. *Class Prompt:* Enhances output representation dimension gating.
  3. *Unified Prompt Template:* Bridges the semantic gap between contrastive pre-training and downstream incremental classification.